In [5]:
import torch
import numpy as np

In [6]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
#load Sigma-Sampler
%run Sigma-Sampler.ipynb
#load Loss-Function
%run Loss-Function-a.ipynb

In [8]:
#compute loss mean function
def compute_loss_mean_a(model, loader, device, num_bins=30, max_batches=None):
    #eval mode
    model.eval()

    all_losses = []
    all_logsnr = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            #break sign
            if max_batches is not None and i >= max_batches:
                break
            x = batch[0].to(device)
            B = x.shape[0]
            #compute per-sample loss
            _, sigma, loss, _ = EDM_loss_a(model, x)
            #compute log-SNR
            logsnr = -2.0 * torch.log(sigma)

            all_losses.append(loss.detach().cpu())
            all_logsnr.append(logsnr.detach().cpu())
    #transform format
    all_losses = torch.cat(all_losses).numpy()
    all_logsnr = torch.cat(all_logsnr).numpy()
    #quantile binning
    bins = np.quantile(all_logsnr, np.linspace(0, 1, num_bins + 1))
    bins[0] -= 1e-6
    bins[-1] += 1e-6

    var_loss_per_bin = []
    mean_loss_per_bin = []
    bin_centers = []
    count_per_bin = []
    #count mean loss to bins
    for i in range(num_bins):
        mask = (all_logsnr >= bins[i]) & (all_logsnr < bins[i+1])
        loss_in_bin = all_losses[mask]
        #delete zero values
        if len(loss_in_bin) > 0:
            var_loss = loss_in_bin.var()
            mean_loss = loss_in_bin.mean()
        else:
            var_loss = np.nan
            mean_loss = np.nan

        var_loss_per_bin.append(var_loss)
        mean_loss_per_bin.append(mean_loss)
        count_per_bin.append(len(loss_in_bin))
        bin_centers.append(0.5 * (bins[i] + bins[i+1]))
    return (
        np.array(bin_centers),
        np.array(var_loss_per_bin),
        np.array(mean_loss_per_bin),
        np.array(count_per_bin)
    )

In [ ]:
#aggregate seeds function
def aggregate_across_seeds(bin_vars_list):
    #stack all seeds
    stacked = np.stack(bin_vars_list, axis=0)
    #filter invalid value
    valid_mask = ~np.isnan(stacked)
    valid_counts = np.sum(valid_mask, axis=0)
    #compute mean & std curve
    mean_curve = np.nanmean(stacked, axis=0)
    std_curve = np.nanstd(stacked, axis=0)
    stderr_curve = std_curve / np.sqrt(np.maximum(valid_counts, 1))
    return mean_curve, stderr_curve